# Feature Importance Interpretation (Tier 1)

## Overview

The top features can be divided into two fundamental signal types:

- **Direct anomaly indicators (low-rating behavior)**
- **Behavioral structure & inconsistency signals**

# Tier 1A — Direct Anomaly Signals

## 1. `min_rating`

**Definition:**  
Minimum rating given by the user.

**Interpretation:**  
- Captures whether the user has ever given an extreme low rating.
- Even a *single* very low rating is highly predictive of anomaly.

**Behavioral Insight:**  
- Anomalous users tend to produce **extreme negative events**.
- This is a **high-signal, low-noise feature**.


## 2. `rating_1_pct`

**Definition:**  
Fraction of ratings equal to 1.

**Interpretation:**  
- Measures how frequently the user gives low ratings.
- Stronger than `rating_0_pct`, indicating “1” is more informative.

**Behavioral Insight:**  
- Anomalies are **consistently negative**, not just occasionally.
- Frequency of low ratings is more important than average rating.

# Tier 1B — Behavioral Structure Signals

## 3. `std_normalized_deviation`

**Definition:**  
Standard deviation of normalized deviations from item mean ratings.

**Interpretation:**  
- Measures how inconsistently a user deviates from consensus.
- High value → user behaves unpredictably across items.

**Behavioral Insight:**  
- Anomalous users are **inconsistent relative to population behavior**.
- Variability matters more than average deviation.

## 4. `user_rating_kurt`

**Definition:**  
Kurtosis of the user’s rating distribution.

**Interpretation:**  
- High kurtosis → ratings concentrated at extremes (e.g., only 1s and 5s).

**Behavioral Insight:**  
- Anomalies exhibit **extreme or bimodal behavior**.
- Indicates non-natural rating patterns.

## 5. `rating_entropy`

**Definition:**  
Entropy of the rating distribution.

**Interpretation:**  
- Low entropy → repetitive behavior (e.g., always same rating)
- High entropy → random/unstructured behavior

**Behavioral Insight:**  
- Anomalies may be:
  - overly consistent (scripted behavior)
  - overly random (no pattern)

## 6. `rating_0_pct`

**Definition:**  
Fraction of ratings equal to 0.

**Interpretation:**  
- Another indicator of extreme low-rating behavior.
- Less important than `rating_1_pct`.

**Behavioral Insight:**  
- Confirms importance of **negative rating patterns**
- Suggests “0” ratings are either rarer or noisier than “1”


# Key Insights

## 1. Dominant Signal

```text
Anomalies are primarily characterized by low-rating behavior

# Experiment 1 (Baseline + Interaction Features)

## Objective
Establish a strong baseline using:
- User-level behavioral features  
- Interaction-level deviation features  

This serves as the reference before adding pattern-based features.


## Dataset
- ~1100 users (1000 normal, 100 anomalous)  
- ~1000 items  
- Input: `(user, item, rating ∈ [0–5])`

## Features Used

### 1. User-Level Features
**Activity**
- `num_unique_items_rated`, `interaction_density`

**Statistics**
- `mean_rating`, `median_rating`, `std_rating`, `min_rating`, `max_rating`

**Distribution**
- `rating_0_pct` → `rating_5_pct`
- `high_rating_ratio`, `low_rating_ratio`

**Shape**
- `rating_entropy`, `user_rating_kurt`, `user_rating_skew`


### 2. Interaction-Level Features

**Item stats (train-only per fold)**
- `item_mean_rating`, `item_std_rating`, `item_popularity`

**Interaction signals**
- `rating_deviation = rating − item_mean`
- `normalized_deviation = rating_deviation / item_std`
- `abs_deviation = |rating_deviation|`

**User aggregation**
- `mean_rating_deviation`, `std_rating_deviation`
- `mean_normalized_deviation`, `std_normalized_deviation`
- `mean_abs_deviation`, `avg_item_popularity`


## Model
- XGBoost  
- Stratified K-Fold (user-level split)  
- Leakage-safe feature computation  


## Performance
- **ROC-AUC ≈ 0.883**  
- **PR-AUC ≈ 0.686**


## Key Learnings

### 1. Strongest Signals
- `min_rating`, `rating_1_pct`, `rating_0_pct`  
→ Detect **low-rating anomalies**

### 2. Distribution Signals
- `entropy`, `kurtosis`, `skew`  
→ Capture **behavioral patterns**


### 3. Interaction Signals
- `std_normalized_deviation`, `mean_abs_deviation`  
→ Capture **inconsistency vs item norms**

## Limitation

Current features capture:
- ✔ average behavior  
- ✔ variability  

Current features capture:
- ✔ average behavior  
- ✔ variability of behavior  

But they do **not explicitly capture event frequency**:

```text
❌ how often a user exhibits extreme anomalous behavior



Motivation:
Add features that explicitly count how often extreme deviations occur,
because anomalies are often driven by these rare events.